# Découpage des observations en jeu d’apprentissage et jeu de tests
Dans un nouveau fichier de script Python, nous allons donc dans un premier temps charger ce fichier et supprimer les lignes pour lesquelles des valeurs sont manquantes comme à la ligne 12 du fichier :

In [14]:
import pandas as pd
#Chargement du dataset avec pour séparateur des tabulations
dataset = pd.read_csv("data/dataset.csv",delimiter='\t')

#Suppression des lignes ayant des valeurs manquantes
dataset = dataset.dropna(axis=0, how='any')

Puis nous allons ensuite extraire les features qui seront source d’apprentissage (données X), c’est-à-dire les colonnes : POINTS_ATTAQUE;POINTS_DEFFENCE;POINTS_ATTAQUE_SPECIALE; POINT_DEFENSE_SPECIALE;POINTS_VITESSE;NOMBRE_GENERATIONS.

In [15]:
#X = on prend toutes les données, mais uniquement les features 5 à 12 
#(ne pas hésiter à ouvrir le fichier afin de bien visualiser les features concernées)
#POINTS_ATTAQUE;POINTS_DEFFENCE;POINTS_ATTAQUE_SPECIALE;POINT_DEFENSE_SPECIALE;POINTS_VITESSE;NOMBRE_GENERATIONS
X = dataset.iloc[:, 5:12].values

Et les features que nous devons prédire à partir de ces données d’apprentissage, c’est-à-dire les valeurs de la colonne POURCENTAGE_DE_VICTOIRE.

In [16]:
#y = on prend uniquement la feature POURCENTAGE_DE_VICTOIRE (17ème feature)
y = dataset.iloc[:, 17].values

Enfin, voici le code associé utilisant la fonction train_test_split du module Scikit-Learn, permettant de créer les jeux de données d’apprentissage et de tests :

In [17]:
#X = on prend toutes les données, mais uniquement les features 4 à 11
# POINTS_ATTAQUE;POINTS_DEFFENCE;POINTS_ATTAQUE_SPECIALE;POINT_DEFENSE_SPECIALE;POINTS_VITESSE;NOMBRE_GENERATIONS
X = dataset.iloc[:, 5:12].values

#y = on prend uniquement la colonne POURCENTAGE_DE_VICTOIRE (16ème feature) 
#les : signifiant "Pour toutes les observations"
y = dataset.iloc[:, 16].values

#Construction du jeu d'entrainement et du jeu de tests
from sklearn.model_selection import train_test_split
X_APPRENTISSAGE, X_VALIDATION, Y_APPRENTISSAGE, Y_VALIDATION = train_test_split(X, y, test_size = 0.2, random_state = 0)

# Algorithme de régression linéaire
Dans notre cas, nous avons plusieurs variables prédictives pour une prédiction. Nous sommes donc dans le cas d’une régression linéaire multiple. Voici comment utiliser l’algorithme de régression linéaire pour réaliser notre apprentissage à l’aide du module Scikit-Learn :

In [18]:
#---- ALGORITHME 1: REGRESSION LINEAIRE -----
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

#Choix de l'algorithme
algorithme = LinearRegression()

#Apprentissage à l'aide de la fonction fit
#Apprentisage effectué sur le jeu de données d'apprentissage
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)

#Realisation de prédictions sur le jeu de tests (validation)
predictions = algorithme.predict(X_VALIDATION)

#Calcul de la précision de l'apprentissage à l'aide de la
#fonction r2_score en comparant les valeurs prédites
#(predictions) et les valeurs attendues (Y_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- REGRESSION LINEAIRE -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- REGRESSION LINEAIRE -----------
>> Precision = 0.8828844617689129
------------------------------------------


# L’arbre de décision appliqué à la régression
Passons à présent à l’utilisation des arbres de décisions en faisant attention à bien utiliser l’algorithme destiné aux problèmes de régression : DecisionTreeRegressor et non celui destiné aux problèmes de classification : DecisionTreeClassifier.

In [19]:
#Choix de l'algorithme
from sklearn.tree import DecisionTreeRegressor
algorithme = DecisionTreeRegressor()
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- ARBRES DE DECISIONS -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- ARBRES DE DECISIONS -----------
>> Precision = 0.7433031766546494
------------------------------------------


# La random forest
Testons à présent l’algorithme des forêts aléatoires (toujours appliqué à la régression) :

In [20]:
#Choix de l'algorithme
from sklearn.ensemble import RandomForestRegressor
algorithme = RandomForestRegressor()
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- FORETS ALEATOIRES -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- FORETS ALEATOIRES -----------
>> Precision = 0.8981178805219476
------------------------------------------


# Sauvegarde du modèle d’apprentissage
Afin de pouvoir utiliser le modèle de prédiction fraîchement entraîné dans une application sans devoir réitérer la phase d’apprentissage, nous devons le sauvegarder.

Pour ce faire, après avoir créé un nouveau répertoire modele dans notre projet, nous allons utiliser le sous-module joblib du module Scikit-learn pour sauvegarder notre modèle d’apprentissage.

In [21]:
import joblib
fichier = 'modele/modele_pokemon.mod'
joblib.dump(algorithme, fichier)

['modele/modele_pokemon.mod']

# Phénomènes de surapprentissage (overfitting) et de sous-apprentissage (underfitting)
Le surapprentissage (overfitting) est un phénomène se traduisant par le fait que la solution est trop adaptée aux données d’apprentissage et ne se généralise pas à de nouvelles données qui lui sont inconnues. Ainsi, si pour un algorithme nous obtenons une précision 99 % sur les données d’apprentissage et que nous obtenons une valeur de 20 % sur les données de tests, il y fort à parier que nous sommes en présence d’un surapprentissage. C’est pourquoi il est conseillé de mesurer la précision à la fois sur les données d’apprentissage et sur les données de validation :

In [22]:
predictions = algorithme.predict(X_VALIDATION)
precision_apprentissage = algorithme.score(X_APPRENTISSAGE,Y_APPRENTISSAGE)
precision = r2_score(Y_VALIDATION, predictions)